In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.RandomResizedCrop(128, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

In [ ]:
import torch
import torch.nn as nn

class MyAmazingCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.features = nn.Sequential(
            # in channels, out channels, kernel size
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
        )
        
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Linear(128, 3)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device

device(type='cuda')

In [11]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

ROOT_DIR = "/content/masters/gmm/lab2/data"
TRAIN_DIR = ROOT_DIR + "/train"
VAL_DIR = ROOT_DIR + "/val"
TEST_DIR = ROOT_DIR + "/test"

train_dataset = ImageFolder(TRAIN_DIR, transform=transform)
test_dataset = ImageFolder(TEST_DIR, transform=transform)
val_dataset = ImageFolder(VAL_DIR, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

In [13]:
def train(model, optimizer, criterion, epochs, save_path):
    train_losses = []
    val_losses = []
    val_accs = []

    best_val_loss = float('inf')
    epochs_no_improve = 0
    patience = 5

    for epoch in range(epochs):

        model.train()
        train_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)    
        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)
        
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)

                predicted = torch.argmax(outputs, dim=1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)
        
        val_loss /= len(val_loader.dataset)
        val_acc = correct / total
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered!")
                break
            
    return train_losses, val_losses, val_accs

In [20]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

def metrics(model):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)

            outputs = model(images)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="macro")
    recall = recall_score(all_labels, all_preds, average="macro")
    f1 = f1_score(all_labels, all_preds, average="macro")
    cm = confusion_matrix(all_labels, all_preds)
    
    print("Accuracy:", acc)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)

    print("Confusion Matrix:")
    print(cm)

In [17]:
import torch.optim as optim

# First experiment
model = MyAmazingCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 25

tl, vl, va = train(model, optimizer, criterion, epochs, ROOT_DIR + "/model-1.pth")

Epoch 1: Train Loss=1.0910, Val Loss=1.0715, Val Acc=0.3600
Epoch 2: Train Loss=1.0636, Val Loss=1.0083, Val Acc=0.4889
Epoch 3: Train Loss=1.0386, Val Loss=0.9880, Val Acc=0.4889
Epoch 4: Train Loss=1.0115, Val Loss=0.9879, Val Acc=0.5378
Epoch 5: Train Loss=1.0033, Val Loss=0.9858, Val Acc=0.5422
Epoch 6: Train Loss=0.9928, Val Loss=0.9451, Val Acc=0.5911
Epoch 7: Train Loss=0.9654, Val Loss=0.9747, Val Acc=0.5556
Epoch 8: Train Loss=0.9497, Val Loss=0.9257, Val Acc=0.5689
Epoch 9: Train Loss=0.9407, Val Loss=0.9182, Val Acc=0.5733
Epoch 10: Train Loss=0.9395, Val Loss=0.9879, Val Acc=0.5156
Epoch 11: Train Loss=0.9234, Val Loss=0.9073, Val Acc=0.5778
Epoch 12: Train Loss=0.9181, Val Loss=0.9056, Val Acc=0.6133
Epoch 13: Train Loss=0.8956, Val Loss=0.9031, Val Acc=0.6133
Epoch 14: Train Loss=0.8719, Val Loss=0.8606, Val Acc=0.6267
Epoch 15: Train Loss=0.8706, Val Loss=0.9030, Val Acc=0.6044
Epoch 16: Train Loss=0.8628, Val Loss=0.9116, Val Acc=0.5467
Epoch 17: Train Loss=0.8725, Val 

In [21]:
metrics(model)

Accuracy: 0.4888888888888889
Precision: 0.5226741095162147
Recall: 0.48888888888888893
F1: 0.47534412205439525
Confusion Matrix:
[[41  5 29]
 [ 9 16 50]
 [ 5 17 53]]


In [18]:
import torch
import torch.nn as nn

class ABetterCNNHopefully(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.features = nn.Sequential(
            # in channels, out channels, kernel size
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
        )
        
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.dropout = nn.Dropout(0.5)

        self.classifier = nn.Linear(128, 3)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.classifier(x)
        return x

In [22]:
# Second experiment
model = ABetterCNNHopefully().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 25

tl, vl, va = train(model, optimizer, criterion, epochs, ROOT_DIR + "/model-2.pth")

Epoch 1: Train Loss=1.0118, Val Loss=0.9799, Val Acc=0.5556
Epoch 2: Train Loss=0.9707, Val Loss=0.9280, Val Acc=0.5556
Epoch 3: Train Loss=0.9469, Val Loss=0.9020, Val Acc=0.5733
Epoch 4: Train Loss=0.9113, Val Loss=0.8881, Val Acc=0.6089
Epoch 5: Train Loss=0.8973, Val Loss=0.9916, Val Acc=0.5067
Epoch 6: Train Loss=0.8853, Val Loss=0.9754, Val Acc=0.5511
Epoch 7: Train Loss=0.8653, Val Loss=1.0036, Val Acc=0.4889
Epoch 8: Train Loss=0.8638, Val Loss=0.8862, Val Acc=0.6267
Epoch 9: Train Loss=0.8781, Val Loss=1.0020, Val Acc=0.5600
Epoch 10: Train Loss=0.8491, Val Loss=0.8715, Val Acc=0.6400
Epoch 11: Train Loss=0.8320, Val Loss=1.1297, Val Acc=0.4978
Epoch 12: Train Loss=0.8260, Val Loss=0.8228, Val Acc=0.6533
Epoch 13: Train Loss=0.8150, Val Loss=0.8913, Val Acc=0.6044
Epoch 14: Train Loss=0.8054, Val Loss=0.8645, Val Acc=0.6133
Epoch 15: Train Loss=0.8163, Val Loss=1.0207, Val Acc=0.5200
Epoch 16: Train Loss=0.8012, Val Loss=0.9616, Val Acc=0.5422
Epoch 17: Train Loss=0.7923, Val 

In [24]:
metrics(model)

Accuracy: 0.5377777777777778
Precision: 0.5255335365853658
Recall: 0.5377777777777778
F1: 0.5026910088984323
Confusion Matrix:
[[69  3  3]
 [26 33 16]
 [28 28 19]]


In [26]:
import torch
import torch.nn as nn

class ImLosingAllHope(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.features = nn.Sequential(
            # in channels, out channels, kernel size
            nn.Conv2d(3, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.dropout = nn.Dropout(0.5)

        self.classifier = nn.Linear(32, 3)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.classifier(x)
        return x

In [ ]:
# Third experiment
model = ImLosingAllHope().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 25

tl, vl, va = train(model, optimizer, criterion, epochs, ROOT_DIR + "/model-3.pth")

Epoch 1: Train Loss=1.1281, Val Loss=1.0703, Val Acc=0.4756
Epoch 2: Train Loss=1.0563, Val Loss=1.0236, Val Acc=0.4978
Epoch 3: Train Loss=1.0550, Val Loss=0.9802, Val Acc=0.5644
Epoch 4: Train Loss=1.0297, Val Loss=0.9716, Val Acc=0.5600
Epoch 5: Train Loss=1.0204, Val Loss=0.9667, Val Acc=0.5556
Epoch 6: Train Loss=1.0146, Val Loss=0.9673, Val Acc=0.5644
Epoch 7: Train Loss=1.0082, Val Loss=0.9580, Val Acc=0.5733
Epoch 8: Train Loss=0.9978, Val Loss=0.9524, Val Acc=0.5822
Epoch 9: Train Loss=1.0110, Val Loss=0.9410, Val Acc=0.5911
Epoch 10: Train Loss=0.9857, Val Loss=0.9322, Val Acc=0.6178
Epoch 11: Train Loss=0.9989, Val Loss=0.9391, Val Acc=0.6000
Epoch 12: Train Loss=0.9827, Val Loss=0.9276, Val Acc=0.6178
Epoch 13: Train Loss=0.9902, Val Loss=0.9183, Val Acc=0.5911
Epoch 14: Train Loss=0.9742, Val Loss=0.9038, Val Acc=0.6133
Epoch 15: Train Loss=0.9792, Val Loss=0.9229, Val Acc=0.5733
Epoch 16: Train Loss=0.9637, Val Loss=0.9267, Val Acc=0.6222
Epoch 17: Train Loss=0.9564, Val 